### Purpose

견적 앱의 이카운트 승인 기록(meta `ECOUNT#REVIEW`)을 글로벌 DB 등록 요청 상태로 옮긴다 — 일회성
(docs/gdb-request-registration-queue.md 4.5, 10장 G5 · 견적 Q8).

review API 는 재고 키만 받으므로, 견적이 내보낸 JSON 을 여기서 DynamoDB 에 직접 기록한다.

**입력 파일** `quote_reviews.json` (견적 `scripts/export_reviews_for_gdb.py` 가 만든다) — 이 노트북 옆에 둔다:

```json
[
  {"kind": "item", "code": "B05012", "status": "pending",
   "requestedBy": "홍길동", "requestedById": "tg:123", "requestedAt": "2026-10-01T02:00:00.000Z", "ref": "quote#Q2026-0123",
   "decidedBy": "", "decidedById": "", "decidedAt": "", "comment": ""},
  {"kind": "customer", "code": "1234567890", "status": "rejected", "...": "..."}
]
```

| 견적 `status` | `regStatus` | `ecountStatus` |
|---|---|---|
| `pending` | `requested` | `pending` |
| `approved` | `approved` | `approved` |
| `rejected` | `rejected` | `excluded` |

- `regRequest.app = "quote"`. 결정이 있으면 `regDecision`(target `registration`)을 남긴다.
- **이미 `regRequest`가 있는 레코드는 건너뛴다** (다시 돌려도 안전). 없는 코드는 목록만 출력.
- 반려를 옮길 때 `active` 는 **바꾸지 않는다** (견적이 이미 쓰고 있는 코드일 수 있음 — 필요하면 웹에서 사용 안 함).
- 셀 3(미리보기)은 쓰지 않는다. 셀 4가 실제로 쓴다.

In [ ]:
# Norton 등이 HTTPS를 가로채는 PC에서 SSL 오류 방지 - boto3 보다 먼저 적용해야 함
import truststore
truststore.inject_into_ssl()

import json
import uuid
from datetime import datetime, timezone
from pathlib import Path

import boto3
from botocore.exceptions import ClientError

boto3.setup_default_session(profile_name='default', region_name='us-east-1')
ddb = boto3.resource('dynamodb')

Settings

In [ ]:
INPUT = Path('quote_reviews.json')
TARGETS = {
    'item': {'table': 'dschemicals', 'key': 'dsids', 'history': 'dschemical_history'},
    'customer': {'table': 'dscustomers', 'key': 'custcd', 'history': 'dscustomer_history'},
}
STATUS_MAP = {'pending': ('requested', 'pending'), 'approved': ('approved', 'approved'), 'rejected': ('rejected', 'excluded')}
DECISIONS = {'approved': 'approve', 'rejected': 'reject'}
BY = 'import:견적 이전'

def now():
    return datetime.now(timezone.utc).isoformat(timespec='milliseconds').replace('+00:00', 'Z')

3. 미리보기 (쓰지 않음)

In [ ]:
entries = json.loads(INPUT.read_text(encoding='utf-8'))
plan, skipped, missing, invalid = [], [], [], []
for e in entries:
    target = TARGETS.get(e.get('kind'))
    if not target or e.get('status') not in STATUS_MAP or not e.get('code'):
        invalid.append(e)
        continue
    record = ddb.Table(target['table']).get_item(Key={target['key']: e['code']}).get('Item')
    if not record:
        missing.append(e['code'])
    elif record.get('regRequest'):
        skipped.append(e['code'])
    else:
        plan.append((e, target, record))
print(f'옮길 것 {len(plan)} · 이미 있음 {len(skipped)} · 없는 코드 {len(missing)} · 형식 오류 {len(invalid)}')
for e, _, record in plan:
    print(f"  {e['kind']:8} {e['code']:12} {record.get('name', '')}  {e['status']} → {STATUS_MAP[e['status']]}")
if missing:
    print('없는 코드:', missing)
if invalid:
    print('형식 오류:', invalid)

4. 기록 (`regRequest` 가 없을 때만 — 조건부 쓰기) + 변경 이력

In [ ]:
done, raced = [], []
for e, target, record in plan:
    reg, ecount = STATUS_MAP[e['status']]
    at = now()
    values = {
        ':reg': reg, ':ecount': ecount, ':now': at, ':by': BY,
        ':request': {'app': 'quote', 'at': e.get('requestedAt') or record.get('createdAt') or at, 'channel': '',
                     'by': e.get('requestedBy', ''), 'byId': e.get('requestedById', ''), 'note': '이전: 견적 이카운트 승인',
                     'ref': e.get('ref', '')},
    }
    sets = 'SET regStatus = :reg, ecountStatus = :ecount, regRequest = :request, updatedAt = :now, updatedBy = :by'
    if e['status'] in DECISIONS:
        values[':decision'] = {'by': e.get('decidedBy', ''), 'byId': e.get('decidedById', ''), 'at': e.get('decidedAt') or at,
                               'comment': e.get('comment', ''), 'target': 'registration', 'decision': DECISIONS[e['status']]}
        sets += ', regDecision = :decision'
    try:
        ddb.Table(target['table']).update_item(
            Key={target['key']: e['code']}, UpdateExpression=sets,
            ConditionExpression='attribute_exists(#k) AND attribute_not_exists(regRequest)',
            ExpressionAttributeNames={'#k': target['key']}, ExpressionAttributeValues=values)
    except ClientError as err:
        if err.response['Error']['Code'] != 'ConditionalCheckFailedException':
            raise
        raced.append(e['code'])
        continue
    ddb.Table(target['history']).put_item(Item={
        target['key']: e['code'], 'at': f'{at}#{uuid.uuid4().hex[:8]}', 'by': BY, 'action': 'import',
        'changes': {'regStatus': ['', reg], 'ecountStatus': ['', ecount]}})
    done.append(e['code'])
print(f'기록 {len(done)} · 그사이 이미 생김 {len(raced)}', raced or '')